# Voting Agent: Big LLM vs Multiple SLMs for SME Agentic Systems

This example demonstrates how to use `VotingAgent` to evaluate different architectural approaches for building agentic systems for Small-Medium Enterprises (SMEs).

**Topic:** Should an SME use one big LLM, multiple small language models (SLMs), or a hybrid approach?

We create multiple agents, each advocating for a different approach, then use `VotingAgent` with `majority_vote` to select the best recommendation.

In [ ]:
from aap_core.agent import BaseAgent
from aap_core.types import AgentMessage, BaseLLMChain
from aap_llamaindex.chain import ChatCausalMultiTurnsChain
from llama_index.llms.ollama import Ollama

## Define the Agent wrapper

A simple agent that executes a chain and returns the response.

In [2]:
class Agent(BaseAgent):
    chain: BaseLLMChain

    def execute(self, message: AgentMessage, **kwargs) -> AgentMessage:
        self.state = "running"
        message = self.chain.invoke(message, **kwargs)
        message.execution_result = "success"
        message.origin = self.card.name
        self.state = "idle"
        return message

## Setup Ollama models

We use different models for each agent to simulate diverse perspectives:
- **qwen3:4b-thinking** — A small thinking model for the Big LLM advocate
- **mistral:7b-instruct** — A small instruct model for the SLM advocate
- **granite3.3:8b** — An 8B model for the Hybrid advocate

In [3]:
def state_callback(state: str):
    print(f"agent state: {state}")

model_big_llm = Ollama(model="qwen3:4b-thinking-2507-q4_K_M", base_url="localhost:11434", request_timeout=400, context_window=32768)
model_slm = Ollama(model="mistral:7b-instruct-v0.3-q4_K_S", base_url="localhost:11434", request_timeout=400, context_window=32768)
model_hybrid = Ollama(model="granite3.3:8b", base_url="localhost:11434", request_timeout=400, context_window=32768)

## Define the three approaches

Each agent advocates for a different architectural approach:
1. **Big LLM Agent** — One large model handles everything
2. **SLM Agent** — Multiple small specialized models
3. **Hybrid Agent** — A mix of both approaches

In [4]:
big_llm_system = """You are an AI architecture consultant specializing in helping Small-Medium Enterprises (SMEs) build AI-powered agentic systems.
Your role is to advocate for using ONE BIG LLM approach.

Arguments to consider:
- A single large model can handle diverse tasks without integration complexity
- Easier to maintain: one model, one pipeline, one monitoring system
- Better at complex reasoning, creative tasks, and handling edge cases
- No need to design task decomposition or routing logic
- Models like Qwen3-30B, Llama-3-70B can handle most SME use cases

Weaknesses to acknowledge:
- Higher inference cost per request
- Slower response times
- Less privacy (sending all data to one powerful model)

Be persuasive but balanced. Give a clear recommendation for SMEs."""

slm_system = """You are an AI architecture consultant specializing in helping Small-Medium Enterprises (SMEs) build AI-powered agentic systems.
Your role is to advocate for using MULTIPLE SMALL LANGUAGE MODELS (SLMs) approach.

Arguments to consider:
- Each SLM can be specialized for a specific task (e.g., one for customer support, one for data analysis)
- Lower inference cost: small models are cheaper to run
- Faster response times for common tasks
- Better data privacy: sensitive data stays with specialized local models
- Easier to update individual models without retraining a large model
- Models like Mistral-7B, Granite-8B are excellent for specific domains

Weaknesses to acknowledge:
- More complex orchestration and routing logic needed
- Harder to handle tasks that require cross-domain reasoning
- More infrastructure to maintain (multiple models, multiple pipelines)

Be persuasive but balanced. Give a clear recommendation for SMEs."""

hybrid_system = """You are an AI architecture consultant specializing in helping Small-Medium Enterprises (SMEs) build AI-powered agentic systems.
Your role is to advocate for a HYBRID approach — mixing big LLMs and small SLMs.

Arguments to consider:
- Use SLMs for common, high-volume tasks (fast, cheap)
- Route complex or ambiguous queries to a big LLM (accurate, capable)
- Best of both worlds: speed + capability
- Gradual migration path: start with SLMs, add big LLM for edge cases
- Cost optimization: 80% of queries handled by cheap SLMs, 20% by expensive LLM
- Example: Mistral-7B for routine support, Qwen3-30B for complex analysis

Weaknesses to acknowledge:
- Most complex architecture to design and maintain
- Need sophisticated routing/classification logic
- Potential inconsistency between model outputs

Be persuasive but balanced. Give a clear recommendation for SMEs."""

In [5]:
user_prompt = "{query}"

big_llm_chain = ChatCausalMultiTurnsChain(
    model=model_big_llm,
    system_prompt=big_llm_system,
    user_prompt_template=user_prompt,
    base_url="localhost:11434",
    request_timeout=300,
    context_window=32768
)

slm_chain = ChatCausalMultiTurnsChain(
    model=model_slm,
    system_prompt=slm_system,
    user_prompt_template=user_prompt,
    base_url="localhost:11434",
    request_timeout=300,
    context_window=32768
)

hybrid_chain = ChatCausalMultiTurnsChain(
    model=model_hybrid,
    system_prompt=hybrid_system,
    user_prompt_template=user_prompt,
    base_url="localhost:11434",
    request_timeout=300,
    context_window=32768
)

In [7]:
# Quick test: verify Ollama connection works from notebook
test_msg = AgentMessage(query="Say 'test' in one word")
test_result = big_llm_chain.invoke(test_msg)
print(f"Test response: {test_result.responses[-1][1][:200]}")

Test response: test


## Create agents and VotingAgent

Each agent has a unique card name (used as the agent identifier in voting). The `VotingAgent` will use BLEU score to compare responses and select the best one.

In [10]:
query = """An SME (Small-Medium Enterprise) with 50 employees wants to build an AI-powered agentic system for customer support and internal knowledge management.
They have a limited budget and limited ML expertise.

Which approach should they choose:
1. One big LLM (e.g., Qwen3-30B) handling all tasks
2. Multiple SLMs (e.g., Mistral-7B, Granite-8B) each specialized for different tasks
3. A hybrid approach mixing both

Consider cost, complexity, performance, and maintainability in your recommendation."""

print(f"Query: {query[:100]}...")

Query: An SME (Small-Medium Enterprise) with 50 employees wants to build an AI-powered agentic system for c...


In [11]:
# Execute each agent to generate their recommendations
big_llm_response = big_llm_agent.execute(AgentMessage(query=query))
slm_response = slm_agent.execute(AgentMessage(query=query))
hybrid_response = hybrid_agent.execute(AgentMessage(query=query))

# Create a message with all responses
message = AgentMessage(query=query)
message.responses.append((big_llm_agent.card.name, big_llm_response.responses[-1][1]))
message.responses.append((slm_agent.card.name, slm_response.responses[-1][1]))
message.responses.append((hybrid_agent.card.name, hybrid_response.responses[-1][1]))

# Now run the voting agent to select the best response
message = voting_agent.execute(message)

print(f"Execution result: {message.execution_result}")
print(f"Total responses: {len(message.responses)}")
print()

for agent_name, response in message.responses:
    print(f"{'='*60}")
    print(f"Agent: {agent_name}")
    print(f"{'='*60}")
    print(response)
    print()

agent state: voting_agent:idle/((big_llm_agent:running)-(slm_agent:idle)-(hybrid_agent:idle))
agent state: voting_agent:idle/((big_llm_agent:idle)-(slm_agent:idle)-(hybrid_agent:idle))
agent state: voting_agent:idle/((big_llm_agent:idle)-(slm_agent:running)-(hybrid_agent:idle))
agent state: voting_agent:idle/((big_llm_agent:idle)-(slm_agent:idle)-(hybrid_agent:idle))
agent state: voting_agent:idle/((big_llm_agent:idle)-(slm_agent:idle)-(hybrid_agent:running))
agent state: voting_agent:idle/((big_llm_agent:idle)-(slm_agent:idle)-(hybrid_agent:idle))
agent state: voting_agent:running/((big_llm_agent:idle)-(slm_agent:idle)-(hybrid_agent:idle))
agent state: voting_agent:idle/((big_llm_agent:idle)-(slm_agent:idle)-(hybrid_agent:idle))
Execution result: success
Total responses: 4

Agent: big_llm_agent
Here's a **clear, actionable, and evidence-based recommendation** tailored specifically for your SME (50 employees, limited budget, limited ML expertise), with all weaknesses addressed head-on:

## Winning recommendation

The last response in `message.responses` is the VotingAgent's selection — the best recommendation among all candidates.

In [12]:
winner_name, winner_response = message.responses[-1]
print(f"🏆 Winning approach: {winner_name}")
print()
print(winner_response)

🏆 Winning approach: voting_agent

Given the constraints of a limited budget and ML expertise, alongside the dual goals of customer support and internal knowledge management, I would recommend a HYBRID approach for this SME. Here's why:

1. **Cost Efficiency**: A hybrid system allows you to utilize smaller, less expensive models (SLMs) for common, high-volume tasks such as answering frequently asked questions or retrieving basic information from the knowledge base. This keeps operational costs low and aligns with your budget constraints. The big LLM (e.g., Qwen3-30B) can then be reserved for more complex queries, ensuring accurate handling of those when they do arise - this is a more cost-effective allocation of resources compared to using a single large model for all tasks.

2. **Performance and Speed**: Smaller models like Mistral-7B or Granite-8B can provide quick responses for routine inquiries, enhancing customer satisfaction with prompt support. On the other hand, a big LLM will e